# 冻结开盘模型的Tiered净动作标签
0拟合、0新策略回测。对当日合法可选择的执行/HOLD计算同一下一日复位后的净价值，比较Tiered与旧25bp比例费用投影。末日缺复位标签保持未知。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/tiered_action_labels_20261014'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
from src.workflow.tiered_action_labels import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

{'experiment': 'tiered_action_labels_20261014',
 'plan_sha256': '96f6528c877a6e70a9c7ae81a3e2aa0538f3c1176ebd4c0b8e30d3a0b461f775',
 'decision': 'diagnostic_only_no_promotion',
 'objective_achieved': False,
 'new_fits': 0,
 'new_strategy_accounts': 0,
 'reused_native_accounts': 3,
 'reference_sessions_per_cost': 944,
 'eligible_rows': 959,
 'scored_rows': 957,
 'unknown_eligible_rows': 2,
 'recorder_id': '2d159d14db804a7a959a5b446ba8a869',
 'experiment_id': '293170839094553539',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/tiered_action_labels_20261014/qlib_recorder',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/tiered_action_labels_20261014/qlib_recorder/293170839094553539/2d159d14db804a7a959a5b446ba8a869',
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013',
 'region': 'us',
 'as_of': '2026-10-09',
 'qlib_docs': 'https://qlib.readthedocs.io/en/latest/component/recorder.html',
 'limits': ['reference own-state one-s

# 冻结动作模型：Tiered净价值诊断

0拟合，0新增策略回测，复用3个Qlib原生账户。每个成本情景保留944个交易区间，合计959个合法可选择动作，其中957个有下一日共同复位标签，末日2个合格动作保留未知。强制减仓、大额转换和不合法HOLD不用于评价分类器选择能力。

逐日重建开盘前持仓、已知净值/高水位、历史收益、风险状态和冻结分类器预测，并断言状态、执行决定、期末股数与现金和原账本一致。当前参考账户均无恢复成交，标签助手对此明确断言，不能用于存在恢复订单的账户。

执行与HOLD分支使用同一起点、真实原价整数股、分支独立月度费率计数，并在下一日09:47按同一参考目标复位，扣两次交易费用。未来复位目标只用于标签，09:46预测不读取它。标签09:49后可用，末日不补造。对照仅把同一状态上的成本改成25bp比例投影，不是原训练标签复现。

## 主要结果

|实际滑点bp|可选择且成熟样本|执行样本|执行平均净优势bp|执行均值95%区块区间bp|HOLD样本若执行的平均优势bp|费率投影标签变号比例|
|---:|---:|---:|---:|---:|---:|---:|
|0|303|164|1.892|[0.311, 3.535]|0.236|3.96%|
|2|323|169|1.680|[0.030, 3.472]|0.042|2.79%|
|5|331|159|1.653|[0.490, 2.788]|0.121|2.72%|

区块重采样保留完整交易日序列及强制/未知行的间隔，63日循环移动块、2000次、固定seed；所报是条件样本均值区间，不是模型选择增益或最终超额区间。分组选择本身基于已冻结预测，未在本卡挑阈值。各组区间不是“执行组减HOLD组”的检验，未做多重研究校正。

2bp全体可选择动作rank IC +0.0852、AUC 0.5359；本卡未计算它们的置信界，不能宣称显著。模型使用balanced-class logistic，输出不能当作校准后的真实胜率。对被选执行的动作，2bp在2025年均值-0.549bp，5bp在2026年均值-0.751bp，存在年度不稳定。

Tiered相对25bp投影降低成本后，约2.7%–4.0%标签变号，执行组均值仍在两种费用投影下为正。因此本证据不支持把原策略落后主要归因于费用标签错配，也不支持反转信号或搜索更好的概率阈值。旧研究的弱条件信息结果使用不同历史/状态域，不应直接与本表相减。

## 决定与下一步

决定：diagnostic_only_no_promotion。这里只确认冻结模型在当前参考状态域存在小幅正的局部执行均值，不能把这些标签加总或复利为可交易策略收益；完整未来风控反馈、分支各自的后续目标和独立前向证据不在标签中。稳定净SOXX超额目标未完成，生产SIM、模型和成本配置均未修改。

停止“费率错配即可解释失败”的假设，不重做已失败的金额回归/可交易域筛选。本轮不支持优先改开盘分类阈值；下一信息需求转向基础分配目标与风控投影：在决策当时可得的状态下，分别记录原目标、投影后目标及持仓约束是否真正绑定，检验现金拖累来自哪一层，再决定是否存在尚未检验的机制。

数据/风险边界继承原账本：复用历史不是独立样本外；当前费率是历史投影，复权价格和实际股息/成交资格未由本卡补足。测试验证共同复位的双边费用及分支费率计数隔离；原生账户未重复回测。

证据：[条件净价值](conditional_value.csv)、[区块区间](block_intervals.csv)、[Recorder对象清单](verification_manifest.json)。
